# Riyadh Services EDA — Neighborhood Lifestyle Profiles

**Objective:** Transform 27K Foursquare places in Riyadh into a structured
neighborhood-level profile by mapping raw venue categories to 14 livability
pillars and spatially joining each point to its district.

**Pipeline:**
1. Load & inspect the raw dataset
2. Define the 14 master category pillars
3. Normalize each venue's category → pillar
4. Spatial join to Riyadh district boundaries
5. Pivot into a per-neighborhood service count matrix
6. Export the final CSV

## 1 — Imports

In [ ]:
import pandas as pd
import geopandas as gpd
import ast
import json

## 2 — Load the Raw Dataset

The CSV contains ~27K Foursquare venues scraped across the Riyadh region.
Each row is one venue with its coordinates, name, category list, and engagement metrics.

In [ ]:
# Load directly from the GitHub repo
url = 'https://raw.githubusercontent.com/AbdulrahmanB-25/Machine_Learning_Competition/main/DataSets/27k_Riyadh_Places.csv'
df_places = pd.read_csv(url)

print(f"Shape: {df_places.shape}")
print(f"Columns: {df_places.columns.tolist()}")
df_places.head()

## 3 — Define the 14 Master Category Pillars

Foursquare categories are granular (200+ unique tags). We collapse them into
14 high-level **livability pillars** using keyword matching.
If a venue's category list contains any of the keywords below, it gets assigned to that pillar.

In [ ]:
NORM_MAP = {
    'dining_cafe':          ['Coffee Shop', 'Café', 'Restaurant', 'Burger Joint', 'Bakery',
                             'Breakfast Spot', 'Juice Bar', 'Dessert Shop', 'Food Truck'],
    'med_facilities':       ['Hospital', 'Medical Center', 'Clinic', 'Emergency Room', 'Dentist', 'Doctor'],
    'health_retail':        ['Pharmacy', 'Drugstore', 'Optical Shop'],
    'fitness_care':         ['Gym', 'Yoga Studio', 'Martial Arts', 'Spa', 'Salon', 'Barber', 'Cosmetics'],
    'edu_primary':          ['Nursery', 'Kindergarten', 'Elementary School', 'Daycare'],
    'edu_higher':           ['High School', 'University', 'College', 'Training Center', 'Language School', 'Library'],
    'religious':            ['Mosque', 'Prayer Room'],
    'essential_retail':     ['Market', 'Grocery', 'Supermarket', 'Laundry', 'Petrol Station', 'Hardware', 'ATM'],
    'parks_green':          ['Park', 'Garden', 'Plaza', 'Promenade', 'Botanical Garden'],
    'sports_play':          ['Soccer Field', 'Padel Court', 'Playground', 'Tennis Court', 'Basketball Court', 'Skate Park'],
    'pedestrian':           ['Walking Trail', 'Pedestrian Street', 'Bike Trail'],
    'resort_rural_retreats':['Farm', 'Campground', 'Stable', 'Resort', 'Theme Park'],
    'gov_civil':            ['Police', 'Post Office', 'Courthouse', 'Embassy', 'Municipality', 'Fire Station'],
    'malls_shopping':       ['Shopping Mall', 'Shopping Center', 'Department Store', 'Outlet Mall', 'Commercial Plaza']
}

print(f"Pillars defined: {len(NORM_MAP)}")
print(f"Total keywords:  {sum(len(v) for v in NORM_MAP.values())}")

## 4 — Normalize Venue Categories → Pillars

Each venue has a `category` column stored as a string representation of a Python list
(e.g. `"['Coffee Shop', 'Breakfast Spot']"`).

**Logic:**
1. Parse the string into an actual list using `ast.literal_eval`
2. For each sub-category, check if any pillar keyword appears (case-insensitive substring match)
3. Assign the **first** matching pillar; if nothing matches → `'other'`

In [ ]:
# Build a reverse lookup: lowercased keyword → pillar name
reverse_map = {sc.lower(): k for k, v in NORM_MAP.items() for sc in v}

def normalize_category(raw):
    """Map a raw category list string to the first matching pillar."""
    try:
        cats = ast.literal_eval(raw)
    except:
        return 'other'
    for cat in cats:
        for keyword, norm_key in reverse_map.items():
            if keyword in cat.lower():
                return norm_key
    return 'other'

df_places['norm_category'] = df_places['category'].apply(normalize_category)

# Show the distribution
print("Pillar distribution:")
print(df_places['norm_category'].value_counts())
print(f"\nMapped:   {(df_places['norm_category'] != 'other').sum()}")
print(f"Unmapped: {(df_places['norm_category'] == 'other').sum()}")

## 5 — Clean Coordinates

Ensure `latitude` and `longitude` are numeric and drop any rows with missing coordinates
(required for the spatial join in the next step).

In [ ]:
df_places['latitude']  = pd.to_numeric(df_places['latitude'],  errors='coerce')
df_places['longitude'] = pd.to_numeric(df_places['longitude'], errors='coerce')

before = len(df_places)
df_places = df_places.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
after = len(df_places)

print(f"Rows before: {before}")
print(f"Rows after:  {after}  (dropped {before - after} with missing coords)")

## 6 — Load Riyadh District Boundaries

We use an official GeoJSON of Saudi district polygons, filtered to Riyadh (`city_id == 3`).
This gives us named neighborhood boundaries for the spatial join.

In [ ]:
geojson_url = (
    'https://raw.githubusercontent.com/AbdulrahmanB-25/Machine_Learning_Competition/'
    '7d9bc540cd6df0e61dec3816c1d990ddb64edfa6/'
    'Saudi-Arabia-Regions-Cities-and-Districts-3.0.0/geojson/districts.geojson'
)

districts = gpd.read_file(geojson_url)
riyadh_districts = districts[districts['city_id'] == 3].copy().to_crs('EPSG:4326')

print(f"Riyadh districts loaded: {len(riyadh_districts)}")
riyadh_districts[['name_en', 'geometry']].head()

## 7 — Spatial Join: Assign Each Venue to a Neighborhood

**Two-pass strategy:**
1. **Exact join (`within`)** — match venues that fall inside a district polygon
2. **Nearest-neighbor fallback** — for venues outside all polygons (e.g. on borders),
   project to UTM Zone 38N and snap to the closest district

This ensures every venue gets a neighborhood label.

In [ ]:
# Convert venues to a GeoDataFrame
gdf = gpd.GeoDataFrame(
    df_places,
    geometry=gpd.points_from_xy(df_places['longitude'], df_places['latitude']),
    crs='EPSG:4326'
)

# Pass 1: Exact spatial join (point-in-polygon)
joined = gpd.sjoin(gdf, riyadh_districts[['name_en', 'geometry']], how='left', predicate='within')
joined['neighborhoods'] = joined['name_en']

print(f"Pass 1 — Matched:   {joined['neighborhoods'].notna().sum()}")
print(f"Pass 1 — Unmatched: {joined['neighborhoods'].isna().sum()}")

In [ ]:
# Pass 2: Nearest-neighbor fallback for unmatched points
missing_mask = joined['neighborhoods'].isna()

if missing_mask.sum() > 0:
    print(f"Running nearest-neighbor join on {missing_mask.sum()} unmatched points...")

    # Project to UTM 38N for accurate distance calculations
    gdf_missing_proj       = gdf[missing_mask].copy().to_crs(epsg=32638)
    riyadh_districts_proj  = riyadh_districts.to_crs(epsg=32638)

    # Nearest join
    nearest = gpd.sjoin_nearest(
        gdf_missing_proj,
        riyadh_districts_proj[['name_en', 'geometry']],
        how='left'
    )

    # A point can tie with 2+ equidistant polygons — keep only the first match
    nearest = nearest[~nearest.index.duplicated(keep='first')]

    # Patch the results back
    joined.loc[missing_mask, 'neighborhoods'] = nearest['name_en'].values
    print(f"After fallback — still unmatched: {joined['neighborhoods'].isna().sum()}")
else:
    print("All points matched on first pass!")

# Write the neighborhood labels back to the main dataframe
df_places['neighborhoods'] = joined['neighborhoods'].values
print(f"\nUnique neighborhoods assigned: {df_places['neighborhoods'].nunique()}")

## 8 — Pivot: Build the Neighborhood Service Profile Matrix

Final reshape:
- Drop venues labeled `'other'` (not in any of the 14 pillars)
- Group by `neighborhoods` × `norm_category` → count
- Ensure all 14 pillar columns exist (some neighborhoods may have 0 for rare pillars)

In [ ]:
# Filter out unmapped categories
df_final = df_places[df_places['norm_category'] != 'other'].copy()
print(f"Venues entering the pivot: {len(df_final)} (dropped {len(df_places) - len(df_final)} 'other')")

# Pivot: rows = neighborhoods, columns = pillars, values = venue count
neighborhood_profiles = (
    df_final
    .groupby(['neighborhoods', 'norm_category'])
    .size()
    .unstack(fill_value=0)
)

# Guarantee all 14 pillar columns are present (fill missing with 0)
neighborhood_profiles = neighborhood_profiles.reindex(columns=NORM_MAP.keys(), fill_value=0)

print(f"\nProfile matrix shape: {neighborhood_profiles.shape}")
neighborhood_profiles.head(10)

## 9 — Export

Save the final neighborhood × pillar count matrix as a CSV.

In [ ]:
output_file = 'Cleaned_Riyadh_Services.csv.csv'
neighborhood_profiles.to_csv(output_file)

print(f"Saved to {output_file}")
print(f"Neighborhoods: {len(neighborhood_profiles)}")
print(f"Pillars:       {len(NORM_MAP)}")

## (Optional) Download in Google Colab

In [ ]:
# Uncomment the lines below if running in Google Colab
# from google.colab import files
# files.download('Cleaned_Riyadh_Services.csv.csv')